<a href="https://colab.research.google.com/github/Namratha36/ATS-Resume/blob/main/Trading_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 1.0 MB/s eta 0:00:00


In [5]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

google_api_key = userdata.get('gemini')
model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a stock trading analyst."
)

In [7]:
import requests
from google.colab import userdata

twelvedata_api_key = userdata.get('TWELVEDATA_API_KEY')

url = "https://api.twelvedata.com/price"
params = {
    "symbol": "AAPL",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)
print(response.json())

{'price': '315.32990'}


In [8]:
from langchain.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    """Get the current market price for a stock. Example: ticker='AAPL'"""
    url = "https://api.twelvedata.com/price"
    params = {"symbol": ticker, "apikey": twelvedata_api_key}
    response = requests.get(url, params=params)
    data = response.json()
    if "price" not in data:
        return f"Could not fetch price for {ticker}: {data.get('message', 'Unknown error')}"
    return f"{ticker} current price: ${float(data['price']):.2f}"

In [9]:
print(get_stock_price.invoke({"ticker": "AAPL"}))

AAPL current price: $315.33


In [10]:
url = "https://api.twelvedata.com/time_series"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "outputsize": "10",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)
print(response.json())

{'meta': {'symbol': 'AAPL', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock'}, 'values': [{'datetime': '2026-07-10', 'open': '314.72000', 'high': '316.91000', 'low': '312.17001', 'close': '315.32001', 'volume': '34109200'}, {'datetime': '2026-07-09', 'open': '310.51001', 'high': '316.53000', 'low': '308.16000', 'close': '316.22000', 'volume': '48124500'}, {'datetime': '2026-07-08', 'open': '311.91000', 'high': '314.82001', 'low': '307.049988', 'close': '313.39001', 'volume': '41323500'}, {'datetime': '2026-07-07', 'open': '315.29001', 'high': '315.48001', 'low': '310.14999', 'close': '310.66000', 'volume': '42490000'}, {'datetime': '2026-07-06', 'open': '307.35999', 'high': '314.20001', 'low': '307', 'close': '312.66000', 'volume': '53590000'}, {'datetime': '2026-07-02', 'open': '294.12000', 'high': '309.42001', 'low': '293.67999', 'close': '308.63000', 'volume': '75352800'}, {'datetime': '2

In [11]:
@tool
def get_price_history(ticker: str) -> str:
    """Get the last 10 days of closing prices for a stock. Example: ticker='AAPL'"""
    url = "https://api.twelvedata.com/time_series"
    params = {
        "symbol": ticker,
        "interval": "1day",
        "outputsize": "10",
        "apikey": twelvedata_api_key
    }
    response = requests.get(url, params=params)
    data = response.json()
    if "values" not in data:
        return f"Could not fetch price history for {ticker}: {data.get('message', 'Unknown error')}"
    prices = [
        f"  {item['datetime']}: ${float(item['close']):.2f}"
        for item in reversed(data["values"])
    ]
    return f"{ticker} closing prices (last 10 days):\n" + "\n".join(prices)

In [12]:
url = "https://api.twelvedata.com/rsi"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "time_period": "14",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)
print(response.json())

{'meta': {'symbol': 'AAPL', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock', 'indicator': {'name': 'RSI - Relative Strength Index', 'series_type': 'close', 'time_period': 14}}, 'values': [{'datetime': '2026-07-10', 'rsi': '62.92718'}, {'datetime': '2026-07-09', 'rsi': '63.84814'}, {'datetime': '2026-07-08', 'rsi': '62.23430'}, {'datetime': '2026-07-07', 'rsi': '60.66124'}, {'datetime': '2026-07-06', 'rsi': '62.43022'}, {'datetime': '2026-07-02', 'rsi': '60.26198'}, {'datetime': '2026-07-01', 'rsi': '50.97136'}, {'datetime': '2026-06-30', 'rsi': '46.91115'}, {'datetime': '2026-06-29', 'rsi': '39.89548'}, {'datetime': '2026-06-26', 'rsi': '41.25063'}, {'datetime': '2026-06-25', 'rsi': '32.20455'}, {'datetime': '2026-06-24', 'rsi': '45.81398'}, {'datetime': '2026-06-23', 'rsi': '47.070787'}, {'datetime': '2026-06-22', 'rsi': '49.89403'}, {'datetime': '2026-06-18', 'rsi': '50.94093'}, {'datetim

In [13]:
@tool
def calculate_rsi(ticker: str) -> str:
    """Calculate the current RSI for a stock to check if it is overbought or oversold. Example: ticker='AAPL'"""
    url = "https://api.twelvedata.com/rsi"
    params = {
        "symbol": ticker,
        "interval": "1day",
        "time_period": "14",
        "apikey": twelvedata_api_key
    }
    response = requests.get(url, params=params)
    data = response.json()
    if "values" not in data:
        return f"Could not fetch RSI for {ticker}: {data.get('message', 'Unknown error')}"
    rsi = float(data["values"][0]["rsi"])
    if rsi > 70:
        signal = "Overbought — price may drop soon"
    elif rsi < 30:
        signal = "Oversold — price may rise soon"
    else:
        signal = "Normal"
    return f"{ticker} RSI(14): {rsi:.1f} → {signal}"

In [14]:
url = "https://api.twelvedata.com/sma"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "time_period": "50",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)
print(response.json())

{'meta': {'symbol': 'AAPL', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock', 'indicator': {'name': 'SMA - Simple Moving Average', 'series_type': 'close', 'time_period': 50}}, 'values': [{'datetime': '2026-07-10', 'sma': '297.76840'}, {'datetime': '2026-07-09', 'sma': '296.87620'}, {'datetime': '2026-07-08', 'sma': '295.90400'}, {'datetime': '2026-07-07', 'sma': '295.057400'}, {'datetime': '2026-07-06', 'sma': '294.31280'}, {'datetime': '2026-07-02', 'sma': '293.52300'}, {'datetime': '2026-07-01', 'sma': '292.67380'}, {'datetime': '2026-06-30', 'sma': '292.24720'}, {'datetime': '2026-06-29', 'sma': '291.86460'}, {'datetime': '2026-06-26', 'sma': '291.49780'}, {'datetime': '2026-06-25', 'sma': '291.15080'}, {'datetime': '2026-06-24', 'sma': '290.82440'}, {'datetime': '2026-06-23', 'sma': '290.14680'}, {'datetime': '2026-06-22', 'sma': '289.47040'}, {'datetime': '2026-06-18', 'sma': '288.74000

In [15]:
@tool
def get_sma(ticker: str) -> str:
    """Get the 50-day and 200-day Simple Moving Averages to check the long-term price trend. Example: ticker='AAPL'"""
    base_params = {
        "symbol": ticker,
        "interval": "1day",
        "apikey": twelvedata_api_key
    }

    response_50 = requests.get(
        "https://api.twelvedata.com/sma",
        params={**base_params, "time_period": "50"}
    )
    response_200 = requests.get(
        "https://api.twelvedata.com/sma",
        params={**base_params, "time_period": "200"}
    )

    data_50 = response_50.json()
    data_200 = response_200.json()

    if "values" not in data_50 or "values" not in data_200:
        return f"Could not fetch SMA for {ticker}"

    sma_50 = float(data_50["values"][0]["sma"])
    sma_200 = float(data_200["values"][0]["sma"])

    if sma_50 > sma_200:
        trend = "Golden Cross — long-term uptrend (Bullish)"
    else:
        trend = "Death Cross — long-term downtrend (Bearish)"

    return f"{ticker} 50-day SMA: ${sma_50:.2f} | 200-day SMA: ${sma_200:.2f} | {trend}"

In [16]:
system_prompt = """You are a stock trading analyst.

When asked to analyse a stock, follow these steps in order:
1. Get the current price → get_stock_price
2. Get the recent price history → get_price_history
3. Calculate RSI → calculate_rsi
4. Get the 50-day and 200-day SMA → get_sma

Then output your analysis in this format:

DECISION: BUY / SELL / HOLD
CONFIDENCE: High / Medium / Low

Current Price: $[price]
Recent trend: [describe the direction from price history]

RSI(14): [value] → [Overbought / Neutral / Oversold]
SMA: [Golden Cross / Death Cross] — [short explanation]

Rationale: [2 sentences explaining the decision based on the signals above]

Do not skip any tool. Always run all four before giving a decision."""

In [17]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[get_stock_price, get_price_history, calculate_rsi, get_sma],
    system_prompt=system_prompt
)

In [18]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Should I buy Apple stock?"}]
})
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'DECISION: BUY\nCONFIDENCE: High\n\nCurrent Price: $315.33\nRecent trend: The stock has shown a consistent upward trend over the last 10 days.\n\nRSI(14): 62.9 → Neutral\nSMA: Golden Cross — The 50-day SMA ($297.77) is above the 200-day SMA ($272.45), indicating a long-term uptrend (Bullish).\n\nRationale: The stock is experiencing a Golden Cross, indicating a strong long-term uptrend. Coupled with a consistent upward movement in recent days, the stock shows strong bullish momentum.', 'extras': {'signature': 'CtMIARFNMg9QkLqMEsgZ8SgqLCvh4znyWuFCE8UIBK6EoS2p92a6dNLDsBhUaC9K5EDWuKuMxLxgvnlqAsxkanT39DdZwl1uUaFmFXFX7MtV66lPV+9OvsodahsP8uOhUEJr8+ila0G8SKyyc5j1pt1PP3Ld43L3fkXrne6LHUdzXNwoI8IzAOuzf0OTyNnVQ3W2ltH9NjE35F98NTxNU5eej2RHEXXjIbUZ6cZZq3Yum9PgKLapmLRVgfIzYEP53sOM6JLKCQ7435P/0k2kS/+sJqY0RKIb2CoPwegsma97dPTxTeXJ2gLbxSXbfTjBCCGS9ApXHdiD8GtRMTYTXgLjFpndbWh9fIExKpIlUKqKG4C80mGPE1cBCW97m06ynab+ouXeB7dac9gDbp1bsxN+8dfBxQPO/J4FsyYNuuEp0DuqXiY554SxX/3VMpT3iNt2dHiOUmy

In [19]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Give me a full analysis of Nvidia."}]
})
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'DECISION: BUY\nCONFIDENCE: High\n\nCurrent Price: $210.95\nRecent trend: The stock has shown a consistent upward trend over the last 10 days, with the price increasing from $192.53 to $210.96.\n\nRSI(14): 57.0 → Neutral\nSMA: Golden Cross — The 50-day SMA ($209.20) is above the 200-day SMA ($191.67), indicating a strong long-term uptrend.\n\nRationale: The Golden Cross signal, combined with a recent positive price trend and a neutral RSI, suggests a strong bullish outlook for Nvidia. These indicators collectively point towards continued upward momentum for the stock.', 'extras': {'signature': 'CsUGARFNMg+c1KuXXwsCaPGIundgp8iNyA5vgd8/QelbHfeZX8Sgz1eZy+lcq9hdZp+ENfEmyQk+iIvks+VR7AB4IYpCJGe4d2ffckzZ7qHrbxIfzCI5kEBSRZvjBftbZSMnT0QrbO2TZHqMgs9fsBf2hb3AOK9XnYCENX1mCxmXdWerWOSAO88Jgkfsw53W8UuKX5NlAKYCMkhaTajE4sF8tY/V0fz5yP63P5+z8WuGuV+QYXwq/h9wB6h56Kq8jRecHObL9KaKMC+nHp3DrgKLsmpGkTwXXidU5NXrdv+mXXhYcljryLb02LlcbePwdTWYLN6kVECJxOMRMMohtxwsoaWiskNvCsa+FIZI0gVpmumTGesp

In [20]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Apple vs Microsoft — which looks technically stronger?"}]
})
print(response["messages"][-1].content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 31.595099187s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT